# Tutorial: From Geometric to Fully Nonlinear Thermal Buckling

In this comprehensive tutorial, we will compare two models of thermal buckling on a slender panel:
1. **Geometric Nonlinearity Only:** The material properties remain constant, and heating is uniform. We will visualize the buckling with an animation.
2. **Fully Nonlinear (Geometric + Material):** The stiffness and thermal expansion change as the panel heats up. We apply the same uniform heating to directly compare how material degradation shifts the critical buckling point.

Finally, we will overlay the maximum deflection of both cases on a single plot to visualize the bifurcation.

# Standard imports

In [ ]:
try:
    from firedrake import *
except ImportError:
    !wget "https://fem-on-colab.github.io/releases/firedrake-install-release-real.sh" -O "/tmp/firedrake-install.sh" && bash "/tmp/firedrake-install.sh"
    from firedrake import *

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import ArtistAnimation
from IPython.display import HTML
from tqdm.auto import tqdm

# Shared Mesh and Function Spaces

We define a slender panel (Length = 10.0, Thickness = 0.1) with 5 elements through the thickness to accurately capture complex bending behaviors. We use a mixed monolithic space $W$ with continuous Galerkin (`CG 1`) elements for both displacement and temperature.

In [ ]:
L, H = 10.0, 0.1
mesh = RectangleMesh(100, 5, L, H)

V_T = FunctionSpace(mesh, "CG", 1)
V_u = VectorFunctionSpace(mesh, "CG", 1)
W = V_u * V_T  # Mixed monolithic space

coords = mesh.coordinates.dat.data.copy()
x, y = SpatialCoordinate(mesh)
d = mesh.topological_dimension()
I = Identity(d)

# Base Properties and Initial Conditions

A perfectly straight mathematical beam under pure compression might compress infinitely without ever snapping. To trigger the physical bifurcation (buckling) in our simulation, we must introduce a **tiny geometric imperfection**. We apply a minuscule sine-wave displacement to the initial state.

In [ ]:
# Time-stepping constants
dt_val = 0.01
dt = Constant(dt_val)
n_steps = 300
time_c = Constant(0.0) # Optimized time tracker for UFL expressions

T_ref = Constant(300.0) 
rho_c = Constant(1.0) 
k = Constant(1.0) 
nu = Constant(0.3)
E_0 = Constant(1e6)
alpha_0 = Constant(2e-4)

# Pre-calculate base Lamé parameters
lmbda_0 = (E_0 * nu) / ((1 + nu) * (1 - 2 * nu))
mu_0 = E_0 / (2 * (1 + nu))
beta_0 = alpha_0 * (3 * lmbda_0 + 2 * mu_0)

# Geometric imperfection to trigger buckling
imperfection = as_vector([0.0, 1e-4 * sin(pi * x / L)])

# Shared Boundary Conditions (Clamped ends)
bc_u_left = DirichletBC(W.sub(0), Constant((0.0, 0.0)), 1)
bc_u_right = DirichletBC(W.sub(0), Constant((0.0, 0.0)), 2)
bcs = [bc_u_left, bc_u_right]

# Part 1: Monolithic Solver Setup (Constant Properties)

## Mathematical Derivation

Because the panel undergoes large rotations during buckling, the small-strain tensor $\epsilon = \frac{1}{2}(\nabla u + \nabla u^T)$ is no longer valid. Instead, we use finite strain kinematics.

### 1. Kinematics
We compute the Deformation Gradient $F$ and the Right Cauchy-Green Tensor $C$:
$$F = I + \nabla u$$
$$C = F^T F$$

The Green-Lagrange Strain Tensor replaces the small-strain tensor:
$$E_{GL} = \frac{1}{2}(C - I)$$

### 2. Constitutive Model (St. Venant-Kirchhoff)
We calculate the Second Piola-Kirchhoff Stress $S$, which relates stresses to the reference configuration, incorporating thermal strains:
$$S = \lambda \text{tr}(E_{GL}) I + 2\mu E_{GL} - \beta (T - T_{ref}) I$$

### 3. Weak Form
To solve the equilibrium equations, we push $S$ forward to the First Piola-Kirchhoff Stress $P$:
$$P = F S$$

The quasi-static mechanical equilibrium is then evaluated against a test function $v_u$:
$$\int_{\Omega} P : \nabla v_u \, dx = 0$$

This is solved monolithically with a transient heat equation driven by a uniform volumetric heat source $r$:
$$\int_{\Omega} \left( \rho_c \frac{T^n - T^{n-1}}{\Delta t} v_T + k \nabla T^n \cdot \nabla v_T - r^n v_T \right) dx = 0$$

In [ ]:
# Initialize states
w1 = Function(W, name="State_1")
w1_old = Function(W, name="State_old_1")
u1, T1 = split(w1)
u1_old, T1_old = split(w1_old)

w1.sub(1).interpolate(T_ref)
w1_old.sub(1).interpolate(T_ref)
w1.sub(0).project(imperfection)
w1_old.sub(0).project(imperfection)

# Kinematics & Stresses
F_def1 = I + grad(u1)
C1 = F_def1.T * F_def1
E_GL1 = 0.5 * (C1 - I)
S1 = lmbda_0 * tr(E_GL1) * I + 2 * mu_0 * E_GL1 - beta_0 * (T1 - T_ref) * I
P1 = F_def1 * S1

# Variational Problem
v_u1, v_T1 = TestFunctions(W)
F_u1 = inner(P1, grad(v_u1)) * dx

T1_rate = (T1 - T1_old) / dt
r1_expr = 50.0 * time_c  # Uniform heating
F_T1 = (rho_c * T1_rate * v_T1 + inner(k * grad(T1), grad(v_T1)) - r1_expr * v_T1) * dx

prob1 = NonlinearVariationalProblem(F_u1 + F_T1, w1, bcs=bcs)
solver1 = NonlinearVariationalSolver(prob1)

# Part 1: Solver Loop and Animation

We setup an animation to visualize the temperature field. Because the panel is extremely slender, we ensure the axis bounds (`set_xlim` and `set_ylim`) and the color scale (`vmin`, `vmax`) are fixed to prevent the plot from flickering as it calculates.

In [ ]:
# Animation Setup
fig, ax = plt.subplots(1, 1, figsize=(10, 3))
ax.set_title("Geometric Buckling (Constant Properties)")
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.set_xlim(-0.5, L + 0.5)
ax.set_ylim(-2.0, 2.0) 
ax.set_aspect('equal') 
frames = []

time_list_1 = []
deflection_list_1 = []
t = 0.0
time_c.assign(t)

T_max_expected = float(T_ref) + 200.0

for step in tqdm(range(n_steps), desc="Solving Part 1 (Geometric)"):
    t += dt_val
    time_c.assign(t) # Update Firedrake Constant for UFL expression
    
    solver1.solve()
    
    u_out, T_out = w1.subfunctions
    abs_max_def = np.max(np.abs(u_out.dat.data[:, 1]))
    time_list_1.append(t)
    deflection_list_1.append(max(abs_max_def, 1e-10))
    
    # Deform mesh for animation
    mesh.coordinates.dat.data[:] = coords + u_out.dat.data[:]
    c1 = tripcolor(T_out, axes=ax, cmap='inferno', vmin=float(T_ref), vmax=T_max_expected)
    frames.append([c1])
    
    # Reset mesh coordinates back to original
    mesh.coordinates.dat.data[:] = coords
    w1_old.assign(w1)

plt.close()
ani = ArtistAnimation(fig, frames, interval=50, blit=True, repeat_delay=1000)
display(HTML(f'<div style="width:100%;">{ani.to_jshtml()}</div>'))

# Part 2: Fully Nonlinear (Geometric + Material)

## Temperature-Dependent Material Properties
Real materials do not maintain constant stiffness or thermal expansion rates as they heat up. Instead of constant values, our stiffness and thermal expansion are now functions of the local temperature $T$. 

The Young's Modulus softens exponentially as temperature rises:
$$E_T = E_0 \exp(-0.005(T - T_{ref}))$$

The thermal expansion coefficient increases linearly with temperature:
$$\alpha_T = \alpha_0 (1.0 + 0.02(T - T_{ref}))$$

We continue to use finite strain kinematics (Green-Lagrange Strain $E_{GL}$ and the Second Piola-Kirchhoff Stress $S$), but now we compute $S$ using the temperature-dependent Lamé parameters $\lambda_T$ and $\mu_T$.

In [ ]:
# Initialize states
w2 = Function(W, name="State_2")
w2_old = Function(W, name="State_old_2")
u2, T2 = split(w2)
u2_old, T2_old = split(w2_old)

w2.sub(1).interpolate(T_ref)
w2_old.sub(1).interpolate(T_ref)
w2.sub(0).project(imperfection)
w2_old.sub(0).project(imperfection)

# Temperature-Dependent Material Properties
E_T = E_0 * exp(-0.005 * (T2 - T_ref)) 
alpha_T = alpha_0 * (1.0 + 0.02 * (T2 - T_ref))

lmbda_T = (E_T * nu) / ((1 + nu) * (1 - 2 * nu))
mu_T = E_T / (2 * (1 + nu))
beta_T = alpha_T * (3 * lmbda_T + 2 * mu_T)

# Kinematics & Stresses
F_def2 = I + grad(u2)
C2 = F_def2.T * F_def2
E_GL2 = 0.5 * (C2 - I)
S2 = lmbda_T * tr(E_GL2) * I + 2 * mu_T * E_GL2 - beta_T * (T2 - T_ref) * I
P2 = F_def2 * S2

# Variational Form Setup
v_u2, v_T2 = TestFunctions(W)
F_u2 = inner(P2, grad(v_u2)) * dx

T2_rate = (T2 - T2_old) / dt
r2_expr = 50.0 * time_c # Uniform heating (same as Part 1)
F_T2 = (rho_c * T2_rate * v_T2 + inner(k * grad(T2), grad(v_T2)) - r2_expr * v_T2) * dx

prob2 = NonlinearVariationalProblem(F_u2 + F_T2, w2, bcs=bcs)
solver2 = NonlinearVariationalSolver(prob2)

# Part 2: Solver Loop

We collect the absolute maximum deflection of the beam at each time step. By strictly enforcing the same uniform heat source `r = 50.0 * time_c`, we can cleanly isolate the impact of the material's structural degradation on the buckling timeline.

In [ ]:
time_list_2 = []
deflection_list_2 = []
t = 0.0
time_c.assign(t)

for step in tqdm(range(n_steps), desc="Solving Part 2 (Fully Nonlinear)"):
    t += dt_val
    time_c.assign(t)
    
    solver2.solve()
    
    u_out_2 = w2.sub(0)
    abs_max_def_2 = np.max(np.abs(u_out_2.dat.data[:, 1]))
    
    time_list_2.append(t)
    deflection_list_2.append(max(abs_max_def_2, 1e-10))
    
    w2_old.assign(w2)

# Overlapping Bifurcation Analysis

By plotting both runs on a semi-log scale, we can clearly identify the critical buckling points. You will observe a flat line of negligible deflection followed by a sharp, vertical spike as the panel snaps out of plane. Notice how the temperature-dependent material degradation (softening modulus and expanding thermal coefficient) causes the fully nonlinear model to buckle significantly earlier than the idealized constant-property model.

In [ ]:
plt.figure(figsize=(10, 6))

plt.semilogy(time_list_1, deflection_list_1, 'b-', linewidth=2, label='Part 1: Constant Properties')
plt.semilogy(time_list_2, deflection_list_2, 'r--', linewidth=2, label='Part 2: Temperature-Dependent')

plt.xlabel('Time (s)')
plt.ylabel('Max Absolute Deflection (m) [Log Scale]')
plt.title('Bifurcation Comparison: Geometric vs Fully Nonlinear')
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.legend()
plt.show()